In [33]:
import os, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter

def load_predict(path, mod_name):
    cols=["contig","position","motif","read_id","label","prob"]
    df=pd.read_csv(path, sep="\t", header=None, usecols=[0,1,2,3,4,5],
                   names=cols, dtype={"contig":str,"position":str,"motif":str,
                                      "read_id":str,"label":str,"prob":float})
    df["position"] = df["position"].astype(int)
    df["mod"] = mod_name
    return df

def to_site_level(df, prob_thres=0.9, cov_thres=6):
    df_f = df.query('label == "mod" and prob >= @prob_thres').copy()
    g = df_f.groupby(["contig","position","motif","mod"], sort=False)["prob"]
    out = g.agg(prob_med="median", n_reads="size").reset_index()
    out = out.query('n_reads >= @cov_thres').copy()
    out["site_id"] = out["contig"] + ":" + out["position"].astype(str) + ":" + out["motif"]
    return out

def build_sets(site_df):
    mods = sorted(site_df["mod"].unique())
    contig_pos = {m:{} for m in mods}
    contig_cov = {m:{} for m in mods}
    contig_motif = {m:{} for m in mods}
    for m in mods:
        sub = site_df.query('mod == @m')[["contig","position","motif","n_reads"]]
        for c, grp in sub.groupby("contig"):
            contig_pos[m][c] = np.sort(grp["position"].values.astype(np.int64))
            contig_cov[m][c] = dict(zip(grp["position"], grp["n_reads"]))
            contig_motif[m][c] = dict(zip(grp["position"], grp["motif"]))
    return mods, contig_pos, contig_cov, contig_motif

def count_overlap_window(a, b, w):
    i = j = cnt = 0
    na, nb = len(a), len(b)
    overlap_idx = []
    while i < na and j < nb:
        if abs(a[i] - b[j]) <= w:
            cnt += 1
            overlap_idx.append((a[i], b[j]))
            i += 1; j += 1
        elif a[i] < b[j] - w:
            i += 1
        else:
            j += 1
    return cnt, overlap_idx

def jaccard_window(mods, contig_pos, contig_cov, contig_motif, w, topk=3):
    n = len(mods)
    J = np.full((n, n), np.nan)
    Omat = np.zeros((n, n), dtype=int)
    top_sites = {}
    cov_pairs = {}

    for i, j in itertools.product(range(n), range(n)):
        Ai = Aj = O = 0
        motif_list = []
        cov_list = []
        keys = set(contig_pos[mods[i]].keys()) | set(contig_pos[mods[j]].keys())
        for c in keys:
            a = contig_pos[mods[i]].get(c, np.array([], dtype=np.int64))
            b = contig_pos[mods[j]].get(c, np.array([], dtype=np.int64))
            Ai += len(a); Aj += len(b)
            if len(a) and len(b):
                cnt, overlaps = count_overlap_window(a, b, w)
                O += cnt
                for (pi, pj) in overlaps:
                    motif_i = contig_motif[mods[i]][c].get(pi)
                    motif_j = contig_motif[mods[j]][c].get(pj)
                    motif_list.append(motif_i if motif_i == motif_j else f"{motif_i}|{motif_j}")
                    cov_i = contig_cov[mods[i]][c].get(pi, 0)
                    cov_j = contig_cov[mods[j]][c].get(pj, 0)
                    cov_list.append((cov_i, cov_j))
        U = Ai + Aj - O
        J[i, j] = O / U if U > 0 else np.nan
        Omat[i, j] = O
        if motif_list:
            top_sites[(i, j)] = [m for m, _ in Counter(motif_list).most_common(topk)]
        if cov_list:
            cov_pairs[(i, j)] = cov_list
    return J, Omat, top_sites, cov_pairs

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import logomaker

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

def plot_half_matrix_advanced(J, mods, Omat,top_sites, cov_pairs=None, A_sizes=None,
                              out_png="matrix.pdf", max_points=200):

    n = len(mods)
    fig, ax = plt.subplots(figsize=(0.8*n+2, 0.8*n+2))

    for i in range(n):
        for j in range(n):
            if i > j:  # 下三角 Jaccard 热图
                val = J[i, j]
                if not np.isnan(val):
                    alpha_val = val * 0.9 + 0.1  # 保证颜色深浅更明显
                    ax.add_patch(plt.Rectangle((j, i), 1, 1,
                                               facecolor="#6F6FF4", alpha=alpha_val))
                    ax.text(j+0.5, i+0.5, f"{val:.2f}",
                            ha="center", va="center", fontsize=7, color="black")
                else:
                    ax.add_patch(plt.Rectangle((j, i), 1, 1,
                                               facecolor="white", edgecolor="grey"))

            elif i < j:  # 上三角 scatter + 拟合线
                if cov_pairs and (i, j) in cov_pairs:
                    covs = np.array(cov_pairs[(i, j)])
                    if len(covs) > 0:
                        # 随机采样
                        if len(covs) > max_points:
                            idx = np.random.choice(len(covs), max_points, replace=False)
                            covs = covs[idx]

                        x_raw, y_raw = covs[:,0], covs[:,1]

                        # 归一化到 [0,1]
                        x = x_raw / (x_raw.max() + 1e-6)
                        y = y_raw / (y_raw.max() + 1e-6)

                        # 映射到格子内部 (左下角为0,0)
                        x = j + 0.1 + 0.8 * x
                        y = i + 0.9 - 0.8 * y   # y=0 在底部

                        # 浅灰散点
                        ax.scatter(x, y, s=6, alpha=0.1, color="lightgrey", edgecolor="none")

                        # 拟合线（深灰）
                        if len(x_raw) > 1:
                            X = x_raw.reshape(-1, 1)
                            model = LinearRegression().fit(X, y_raw)
                            x_line = np.linspace(0, x_raw.max(), 50).reshape(-1, 1)
                            y_line = model.predict(x_line)

                            # 归一化并映射到格子
                            x_line_norm = x_line.flatten() / (x_raw.max() + 1e-6)
                            y_line_norm = y_line / (y_raw.max() + 1e-6)
                            x_line_plot = j + 0.1 + 0.8 * x_line_norm
                            y_line_plot = i + 0.9 - 0.8 * y_line_norm

                            ax.plot(x_line_plot, y_line_plot, color="dimgray", linewidth=1)


            else:  # 对角线：Top 3 motifs + 覆盖度分布
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor="lightgrey"))

                # Top 3 motifs
                if top_sites and (i, i) in top_sites:
                    motifs = top_sites[(i, i)][:3]  # 取前三个
                    for k, motif in enumerate(motifs):
                        motif = motif.replace("T", "U")
                        ax.text(j+0.5, i+0.2 + k*0.25, motif,
                                ha="center", va="center",
                                fontsize=6, fontname="Arial", color="darkgreen")


    # 去掉默认刻度
    ax.set_xticks([]); ax.set_yticks([])

    for j, mod in enumerate(mods):
        ax.text(j+0.5, n+0.3, mod, ha="right", va="center",
                fontsize=9, fontname="Arial", rotation=45)

    # 在左侧写 y 标签（居中，倾斜 45°）
    for i, mod in enumerate(mods):
        ax.text(0, i+0.5, mod, ha="right", va="center",
                fontsize=9, fontname="Arial", rotation=45)


    ax.set_xlim(0, n); ax.set_ylim(0, n)
    ax.invert_yaxis()
    ax.set_title("Co-occurrence modification",fontname="Arial", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=1200)
    plt.close()



def run(mod2file, outdir, prob_thres=0.9, cov_thres=6, window_nt=5):
    os.makedirs(outdir, exist_ok=True)
    dfs = [load_predict(p, m) for m, p in mod2file]
    df = pd.concat(dfs, ignore_index=True)

    site_df = to_site_level(df, prob_thres=prob_thres, cov_thres=cov_thres)
    mods, contig_pos, contig_cov, contig_motif = build_sets(site_df)

    J, Omat, top_sites, cov_pairs = jaccard_window(mods, contig_pos, contig_cov, contig_motif, window_nt)

    plot_half_matrix_advanced(
        J, mods, Omat,top_sites, cov_pairs=cov_pairs,
        out_png=os.path.join(outdir, f"cooccur_win{window_nt}_scatter.pdf")
    )


In [35]:
mod2file = [
    ("hm5C","/mnt/sunxh/humancell_dataset/predict.hetwt.hm5c.tsv"),
    ("Inosine","/mnt/sunxh/humancell_dataset/predict.hetwt.inosine.tsv"),
    ("m1A","/mnt/sunxh/humancell_dataset/predict.hetwt.m1a.tsv"),
    ("m5C","/mnt/sunxh/humancell_dataset/predict.hetwt.m5c.tsv"),
    ("m6A","/mnt/sunxh/humancell_dataset/predict.hetwt.m6a.tsv"),
    ("m7G","/mnt/sunxh/humancell_dataset/predict.hetwt.m7g.tsv"),
    ("Psu","/mnt/sunxh/humancell_dataset/predict.hetwt.psu.tsv"),
]
outdir="/mnt/sunxh/humancell_dataset/vis_multi_mod_cooccur"
os.makedirs(outdir, exist_ok=True)
dfs = [load_predict(p, m) for m, p in mod2file]
df = pd.concat(dfs, ignore_index=True)

site_df = to_site_level(df, prob_thres=0.9, cov_thres=6)
mods, contig_pos, contig_cov, contig_motif = build_sets(site_df)

J, Omat, top_sites, cov_pairs = jaccard_window(mods, contig_pos, contig_cov, contig_motif, 5)


In [37]:
def plot_half_matrix_advanced(J, mods, Omat,top_sites, cov_pairs=None, A_sizes=None,
                              out_png="matrix.pdf", max_points=200):

    n = len(mods)
    fig, ax = plt.subplots(figsize=(0.8*n+2, 0.8*n+2))

    for i in range(n):
        for j in range(n):
            if i > j:  # 下三角 Jaccard 热图
                val = J[i, j]
                if not np.isnan(val):
                    alpha_val = val * 0.9 + 0.1  # 保证颜色深浅更明显
                    ax.add_patch(plt.Rectangle((j, i), 1, 1,
                                               facecolor="#6F6FF4", alpha=alpha_val))
                    ax.text(j+0.5, i+0.5, f"{val:.2f}",
                            ha="center", va="center", fontsize=7, color="black")
                else:
                    ax.add_patch(plt.Rectangle((j, i), 1, 1,
                                               facecolor="white", edgecolor="grey"))

            elif i < j:  # 上三角 scatter + 拟合线
                if cov_pairs and (i, j) in cov_pairs:
                    covs = np.array(cov_pairs[(i, j)])
                    if len(covs) > 0:
                        # 随机采样
                        if len(covs) > max_points:
                            idx = np.random.choice(len(covs), max_points, replace=False)
                            covs = covs[idx]

                        x_raw, y_raw = covs[:,0], covs[:,1]

                        # 归一化到 [0,1]
                        x = x_raw / (x_raw.max() + 1e-6)
                        y = y_raw / (y_raw.max() + 1e-6)

                        # 映射到格子内部 (左下角为0,0)
                        x = j + 0.1 + 0.8 * x
                        y = i + 0.9 - 0.8 * y   # y=0 在底部

                        # 浅灰散点
                        ax.scatter(x, y, s=6, alpha=0.9, color="darkblue", edgecolor="none")

                        # 拟合线（深灰）
                        if len(x_raw) > 1:
                            X = x_raw.reshape(-1, 1)
                            model = LinearRegression().fit(X, y_raw)
                            x_line = np.linspace(0, x_raw.max(), 50).reshape(-1, 1)
                            y_line = model.predict(x_line)

                            # 归一化并映射到格子
                            x_line_norm = x_line.flatten() / (x_raw.max() + 1e-6)
                            y_line_norm = y_line / (y_raw.max() + 1e-6)
                            x_line_plot = j + 0.1 + 0.8 * x_line_norm
                            y_line_plot = i + 0.9 - 0.8 * y_line_norm

                            ax.plot(x_line_plot, y_line_plot, color="dimgray", linewidth=1)


            else:  # 对角线：Top 3 motifs + 覆盖度分布
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor="lightgrey"))

                # Top 3 motifs
                if top_sites and (i, i) in top_sites:
                    motifs = top_sites[(i, i)][:3]  # 取前三个
                    for k, motif in enumerate(motifs):
                        motif = motif.replace("T", "U")
                        ax.text(j+0.5, i+0.2 + k*0.25, motif,
                                ha="center", va="center",
                                fontsize=6, fontname="Arial", color="darkgreen")


    # 去掉默认刻度
    ax.set_xticks([]); ax.set_yticks([])

    for j, mod in enumerate(mods):
        ax.text(j+0.5, n+0.3, mod, ha="right", va="center",
                fontsize=9, fontname="Arial", rotation=45)

    # 在左侧写 y 标签（居中，倾斜 45°）
    for i, mod in enumerate(mods):
        ax.text(0, i+0.5, mod, ha="right", va="center",
                fontsize=9, fontname="Arial", rotation=45)


    ax.set_xlim(0, n); ax.set_ylim(0, n)
    ax.invert_yaxis()
    ax.set_title("Co-occurrence modification",fontname="Arial", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=1200)
    plt.close()
plot_half_matrix_advanced(
    J, mods, Omat,top_sites, cov_pairs=cov_pairs,
    out_png=os.path.join(outdir, f"cooccur_win{5}_scatter.pdf")
)
